In [15]:
%load_ext autoreload
%autoreload 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from gensim.parsing.preprocessing import split_alphanum
from gensim.parsing.preprocessing import strip_non_alphanum
from gensim.parsing.preprocessing import strip_punctuation
from gensim.parsing.preprocessing import strip_tags
from gensim.parsing.preprocessing import strip_multiple_whitespaces
from pandas.core.common import flatten

In [17]:
from __future__ import print_function
import ruamel.yaml as ry
import pandas as pd

In [18]:
import os
import sys
sys.path.append('../')

In [19]:
from src.data.make_dataset import DataProcessing, DataToAzure

In [20]:
config_file = "/home/ml/Documents/BI_Help/config/config.yml"

In [21]:
def build_configuration(config_file):
    """
    Build the operation configuration dict
    :param config_file: is the path to the yaml config_file
    :type: string
    :return: config: global configuration
    :rtype dict
    """
    # yaml config
    
    with open(config_file, 'r') as stream:
        try:
            return ry.safe_load(stream)
        except ry.YAMLError as exc:
            print(exc)

In [22]:
config = build_configuration(config_file)

In [23]:
obj_df_loaded = DataProcessing(config)

## Data exploration

In [24]:
%%time
df_merged = obj_df_loaded.return_merged_df()

b'Skipping line 2189: expected 367 fields, saw 369\n'
b'Skipping line 27321: expected 367 fields, saw 369\nSkipping line 27322: expected 367 fields, saw 369\n'


CPU times: user 1min 34s, sys: 5.84 s, total: 1min 40s
Wall time: 1min 40s


In [25]:
df_merged

,code_correction,TXZ01 Short Text,Material grp desc. 2
0,42500,PIGNON 36 DENTS,"Comp. Méca. (moteur, actionneur, vérin, roulem..."
2,42500,"GAINE RETRACTABLE DIAM.1,2MM TRANSPARENT","Comp. Méca. (moteur, actionneur, vérin, roulem..."
3,44100,ACC_CON PROTECTEUR CARTE EMB,connecteurs circulaires et accessoires ...
4,44100,CONNECTEUR HE804ES53Z1A221,connecteurs circulaires et accessoires ...
5,44100,CONNECTEUR 65039-27P,connecteurs circulaires et accessoires ...
...,...,...,...
365983,64200,ESSAI CEM / CTH RAFALE,"Prest. Essais et Tests (analyses, contrôles, E..."
366003,60600,SILOUET - Data management,Prest en AT Ing Syst Hors IVVQ ...
366017,15201A,GDM V3 Livraison en recette,ST Dév d'appli ou infra interne (Fnr hors Gpe)...
366018,15201A,GDM V3 Mise en production,ST Dév d'appli ou infra interne (Fnr hors Gpe)...


In [26]:
df_merged['TXZ01 Short Text'] = df_merged['TXZ01 Short Text'].apply(lambda x : x.split(' '))
df_for_exploration = df_merged.groupby(['code_correction', 'Material grp desc. 2'])['TXZ01 Short Text'].apply(list).reset_index(name='text_grouped')
df_for_exploration['text_grouped_flatten'] = df_for_exploration['text_grouped'].apply(lambda x : list(flatten(x)))
df_for_exploration['occurence_words'] = df_for_exploration['text_grouped_flatten'].apply(lambda x : len(x))
df_for_exploration['occurence_code'] = df_for_exploration['text_grouped'].apply(lambda x : len(x))

In [27]:
df_for_exploration

,code_correction,Material grp desc. 2,text_grouped,text_grouped_flatten,occurence_words,occurence_code
0,01100,séminaire congrès convention presta hotelière ...,"[[PRESTATION, TRAITEUR, POUR, 2012], [Speakerw...","[PRESTATION, TRAITEUR, POUR, 2012, Speakerwise...",3281,655
1,01200,Frais d'inscription aux foires et expositions ...,"[[COMP, AFF, MONITEUR, TUBE, M12-120GY], [ECRO...","[COMP, AFF, MONITEUR, TUBE, M12-120GY, ECROU, ...",510,134
2,01300,Prestations annexes aux foires et expositions ...,"[[Screenshot], [Fourniture, bo, tier, TOP, AXY...","[Screenshot, Fourniture, bo, tier, TOP, AXYZ, ...",487,129
3,013002TAV,Electricité ...,[[test]],[test],1,1
4,01300A,TGS Participation Salon ...,"[[Salons, TGS, 2016], [2018, Exhibitions, Even...","[Salons, TGS, 2016, 2018, Exhibitions, Events,...",7,3
...,...,...,...,...,...,...
222,92001,moyens de production de front end et de back e...,"[[Contr, le, metrologique, table, EVO, 10], [C...","[Contr, le, metrologique, table, EVO, 10, Cont...",99,19
223,92040,Maintenance de moyens de production front end ...,"[[pieces, detachees, Qualmark], [EXCAL, 1421, ...","[pieces, detachees, Qualmark, EXCAL, 1421, TE,...",1778,403
224,95100,équipements aéronautiques ...,"[[STANDBY, CONTROL, PANEL], [VHF, TRANCEIVER, ...","[STANDBY, CONTROL, PANEL, VHF, TRANCEIVER, 8.3...",724,251
225,9510010,Groupe 95100 classe valo. 8010 ...,"[[Keyboard, (KBD), MFD5x4, (E24273BA)], [Displ...","[Keyboard, (KBD), MFD5x4, (E24273BA), Display,...",17,4


In [29]:
df_for_exploration['rapport'] = df_for_exploration['occurence_words']/df_for_exploration['occurence_code']

In [30]:
df_for_exploration

,code_correction,Material grp desc. 2,text_grouped,text_grouped_flatten,occurence_words,occurence_code,rapport
0,01100,séminaire congrès convention presta hotelière ...,"[[PRESTATION, TRAITEUR, POUR, 2012], [Speakerw...","[PRESTATION, TRAITEUR, POUR, 2012, Speakerwise...",3281,655,5.009160
1,01200,Frais d'inscription aux foires et expositions ...,"[[COMP, AFF, MONITEUR, TUBE, M12-120GY], [ECRO...","[COMP, AFF, MONITEUR, TUBE, M12-120GY, ECROU, ...",510,134,3.805970
2,01300,Prestations annexes aux foires et expositions ...,"[[Screenshot], [Fourniture, bo, tier, TOP, AXY...","[Screenshot, Fourniture, bo, tier, TOP, AXYZ, ...",487,129,3.775194
3,013002TAV,Electricité ...,[[test]],[test],1,1,1.000000
4,01300A,TGS Participation Salon ...,"[[Salons, TGS, 2016], [2018, Exhibitions, Even...","[Salons, TGS, 2016, 2018, Exhibitions, Events,...",7,3,2.333333
...,...,...,...,...,...,...,...
222,92001,moyens de production de front end et de back e...,"[[Contr, le, metrologique, table, EVO, 10], [C...","[Contr, le, metrologique, table, EVO, 10, Cont...",99,19,5.210526
223,92040,Maintenance de moyens de production front end ...,"[[pieces, detachees, Qualmark], [EXCAL, 1421, ...","[pieces, detachees, Qualmark, EXCAL, 1421, TE,...",1778,403,4.411911
224,95100,équipements aéronautiques ...,"[[STANDBY, CONTROL, PANEL], [VHF, TRANCEIVER, ...","[STANDBY, CONTROL, PANEL, VHF, TRANCEIVER, 8.3...",724,251,2.884462
225,9510010,Groupe 95100 classe valo. 8010 ...,"[[Keyboard, (KBD), MFD5x4, (E24273BA)], [Displ...","[Keyboard, (KBD), MFD5x4, (E24273BA), Display,...",17,4,4.250000


In [31]:
df_for_exploration['rapport'].describe()

count    227.000000
mean       4.261163
std        1.154501
min        1.000000
25%        3.593158
50%        4.250000
75%        4.956522
max        9.000000
Name: rapport, dtype: float64

### Visualisation 

In [ ]:
df_for_exploration.sort_values('len_text', ascending = True).plot.barh(x='MATKLMaterialGroupCorrige', y='len_text', figsize = (30,50), title='lenght of text by code')

In [ ]:
df_number_code .sort_values('number_of_code', ascending = True).plot.barh(x='MATKLMaterialGroupCorrige', y='number_of_code', figsize = (30,50), title='number by code')

In [ ]:
df_merge [df_merge ['MATKLMaterialGroupCorrige']=='16021A']

In [ ]:
df_merge[df_merge ['MATKLMaterialGroupCorrige']=='47200']

In [ ]:
list_of_code = df_merge['MATKLMaterialGroupCorrige'].unique()

In [ ]:
def plot_words_frequency(code):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(30, 15))  # the size you want
    df_test = list(df_for_exploration[df_for_exploration['MATKLMaterialGroupCorrige'] == code].list_of_words)
    fdist = FreqDist(df_test[0])
    fdist.plot(100, cumulative=False)

In [ ]:
list_of_code[0]

In [ ]:
plot_words_frequency(list_of_code[1])

In [ ]:
from wordcloud import WordCloud

In [ ]:
text = ' ' .join(list(df_merge['TXZ01ShortText']))

In [ ]:
wordcloud = WordCloud().generate(text)


In [ ]:
wordcloud = WordCloud(max_font_size=40).generate(text)
plt.figure(figsize =(30,20))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

### Data cleaning